# File Operations and Image Processing

This enhanced notebook demonstrates file upload, processing, and retrieval in the ADEPT ecosystem,
with special focus on image/PNG file handling based on the validated test suite.

## Security Architecture

All file operations route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net` — it cannot reach internal services directly.

## Features Covered
- Upload files via nginx_proxy API (CSV, JSON, Images)
- Process images with AI agents (PNG, JPEG)
- File registration and metadata management
- Download processed results
- Security validations (authentication, path traversal)
- Complete image processing lifecycle

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

## 1. Setup and Authentication

In [ ]:
import os
import json
import httpx
import asyncio
import pandas as pd
from pathlib import Path
import openai
import base64
from PIL import Image, ImageDraw, ImageFont
import io
import tempfile

# Load authentication
def load_auth():
    with open("/tmp/adept_auth.json", "r") as f:
        return json.load(f)

def load_config():
    with open("/tmp/adept_config.json", "r") as f:
        return json.load(f)

auth = load_auth()
config = load_config()

API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))
NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')
ACCESS_TOKEN = auth['access_token']

# Initialize OpenAI client — routes through nginx_proxy
client = openai.OpenAI(
    base_url=API_BASE_URL,
    api_key=ACCESS_TOKEN,
    http_client=httpx.Client(verify=False)
)

print(f"Connected to: {API_BASE_URL}")
print(f"User: {auth['username']}")
print(f"Routing: nginx_proxy (TLS + JWT)")

## 2. Create Sample Files (Including Images)

In [ ]:
# Create sample CSV data
csv_data = pd.DataFrame({
    'sample_id': ['S001', 'S002', 'S003', 'S004', 'S005'],
    'temperature': [37.2, 38.1, 36.9, 37.5, 38.3],
    'concentration': [2.5, 3.1, 2.8, 2.9, 3.3],
    'ph_level': [7.2, 7.4, 7.1, 7.3, 7.5],
    'reaction_time': [120, 135, 125, 130, 140]
})

csv_file = "/tmp/experiment_data.csv"
csv_data.to_csv(csv_file, index=False)
print(f"✅ Created: {csv_file}")

# Create a sample PNG image
def create_test_image(text="TEST", size=(100, 100), color='blue'):
    """Create a test PNG image."""
    img = Image.new('RGB', size, color=color)
    draw = ImageDraw.Draw(img)
    
    # Add text to make it identifiable
    text_bbox = draw.textbbox((0, 0), text)
    text_width = text_bbox[2] - text_bbox[0]
    text_height = text_bbox[3] - text_bbox[1]
    text_x = (size[0] - text_width) // 2
    text_y = (size[1] - text_height) // 2
    draw.text((text_x, text_y), text, fill='white')
    
    return img

# Create and save test image
test_image = create_test_image("SAMPLE", (200, 200), 'navy')
png_file = "/tmp/test_image.png"
test_image.save(png_file, 'PNG')
print(f"✅ Created: {png_file}")

# Display the image
test_image

## 3. Enhanced File Upload with Type Detection

In [ ]:
async def upload_file(file_path: str, purpose: str = "assistants") -> dict:
    """Upload a file via nginx_proxy with proper content type detection."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        file_ext = Path(file_path).suffix.lower()
        content_types = {
            '.png': 'image/png',
            '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg',
            '.csv': 'text/csv',
            '.json': 'application/json',
            '.txt': 'text/plain'
        }
        content_type = content_types.get(file_ext, 'application/octet-stream')
        
        with open(file_path, "rb") as f:
            files = {"file": (Path(file_path).name, f, content_type)}
            data = {"purpose": purpose}
            headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
            
            response = await http_client.post(
                f"{API_BASE_URL}/files",
                files=files,
                data=data,
                headers=headers
            )
            
            if response.status_code in [200, 201]:
                return response.json()
            else:
                return {"error": f"Upload failed: {response.status_code} - {response.text}"}

# Upload all files
print("Uploading files...\n")

csv_upload = await upload_file(csv_file)
if "error" not in csv_upload:
    print(f"CSV uploaded: {csv_upload['id']}")
    print(f"   Filename: {csv_upload['filename']}")
    print(f"   Size: {csv_upload['bytes']} bytes")
else:
    print(f"CSV upload failed: {csv_upload['error']}")

png_upload = await upload_file(png_file)
if "error" not in png_upload:
    print(f"\nPNG uploaded: {png_upload['id']}")
    print(f"   Filename: {png_upload['filename']}")
    print(f"   Size: {png_upload['bytes']} bytes")
else:
    print(f"PNG upload failed: {png_upload['error']}")

## 4. Process PNG Image with Agent (Sandbox Processing)

In [ ]:
# Process the PNG image through sandbox MCP server
if "error" not in png_upload:
    # Directive instructions for image processing (based on test suite)
    image_processing_prompt = f"""I've uploaded a PNG file with ID: {png_upload['id']}

Please use the execute_code tool to process this image:
1. Load the PNG from /app/data/uploaded_files/{png_upload['id']}
2. Add a 10-pixel red border around the image
3. Add yellow text "PROCESSED" at position (20, 20)
4. Save the processed image
5. Return the processed image file path or ID

Use PIL/Pillow for image processing."""
    
    print("Sending image for processing...\n")
    
    response = client.chat.completions.create(
        model="agentic-framework:scientific-agent-v1",
        messages=[{"role": "user", "content": image_processing_prompt}],
        temperature=0.7,
        max_tokens=2000,
        stream=False
    )
    
    print("Agent Processing Response:")
    print("=" * 60)
    print(response.choices[0].message.content)
    print("=" * 60)

## 5. Verify File Serving and List Files

In [ ]:
async def list_files(limit: int = 10):
    """List all uploaded files via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        response = await http_client.get(
            f"{API_BASE_URL}/files",
            headers=headers,
            params={"limit": limit}
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Failed to list files: {response.status_code} - {response.text}"}

files_list = await list_files()

if "error" not in files_list:
    print(f"Files in your account: {len(files_list.get('data', []))}\n")
    
    our_files = []
    for file in files_list.get('data', []):
        if file['id'] in [csv_upload.get('id'), png_upload.get('id')]:
            our_files.append(file)
            print(f"Found: {file['filename']}")
            print(f"  ID: {file['id']}")
            print(f"  Size: {file['bytes']} bytes")
            print(f"  Created: {file['created_at']}")
            print()
else:
    print(f"Error: {files_list['error']}")

## 6. Download and Validate PNG Files

In [ ]:
async def download_and_validate_png(file_id: str, save_path: str = None):
    """Download PNG file via nginx_proxy and validate it's a valid image."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        
        response = await http_client.get(
            f"{API_BASE_URL}/files/{file_id}/content",
            headers=headers
        )
        
        if response.status_code == 404:
            response = await http_client.get(
                f"{API_BASE_URL}/files/{file_id}",
                headers=headers
            )
        
        if response.status_code == 200:
            png_data = response.content
            
            if len(png_data) > 8 and png_data[:8] == b'\x89PNG\r\n\x1a\n':
                print(f"Valid PNG file downloaded ({len(png_data)} bytes)")
                
                try:
                    img = Image.open(io.BytesIO(png_data))
                    print(f"   Image details: {img.size[0]}x{img.size[1]} pixels, {img.mode} mode")
                    
                    if save_path:
                        with open(save_path, "wb") as f:
                            f.write(png_data)
                        print(f"   Saved to: {save_path}")
                    
                    return {"success": True, "image": img, "data": png_data}
                except Exception as e:
                    return {"error": f"Failed to validate PNG: {e}"}
            else:
                return {"error": "Invalid PNG header"}
        else:
            return {"error": f"Download failed: {response.status_code}"}

if "error" not in png_upload:
    result = await download_and_validate_png(
        png_upload['id'], 
        "/tmp/downloaded_original.png"
    )
    
    if result.get('success'):
        print("\nOriginal image downloaded successfully!")
        result['image']

## 7. Security Validation Tests

In [ ]:
async def test_security_validations(file_id: str):
    """Test security aspects of file handling via nginx_proxy."""
    print("Testing file security...\n")
    
    # Test 1: Unauthenticated access should fail
    print("Test 1: Unauthenticated access")
    async with httpx.AsyncClient(verify=False) as client:
        response = await client.get(f"{API_BASE_URL}/files/{file_id}")
        if response.status_code in [401, 403]:
            print("  PASS: Unauthenticated access blocked")
        else:
            print(f"  FAIL: Unexpected response: {response.status_code}")
    
    # Test 2: Directory traversal should be blocked
    print("\nTest 2: Directory traversal protection")
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    malicious_paths = [
        "../../../etc/passwd",
        "..\\..\\..\\windows\\system32\\config\\sam",
        "%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2fpasswd"
    ]
    
    async with httpx.AsyncClient(verify=False) as client:
        for path in malicious_paths:
            response = await client.get(
                f"{API_BASE_URL}/files/{path}",
                headers=headers
            )
            if response.status_code in [400, 403, 404]:
                print(f"  PASS: Blocked: {path[:30]}... ({response.status_code})")
            else:
                print(f"  FAIL: Not blocked: {path} ({response.status_code})")
    
    print("\nSecurity validation complete")

if "error" not in png_upload:
    await test_security_validations(png_upload['id'])

## 8. Generate Data Visualization

In [ ]:
# Ask agent to generate a plot from the CSV data
if "error" not in csv_upload:
    plot_prompt = f"""Using the CSV file {csv_upload['filename']} (ID: {csv_upload['id']}),
please create a correlation matrix heatmap showing relationships between all numerical variables.
Use matplotlib and seaborn to create a professional visualization.
Save the plot as a PNG file and let me know the file ID so I can download it."""
    
    print("Requesting data visualization...\n")
    
    plot_response = client.chat.completions.create(
        model="agentic-framework:scientific-agent-v1",
        messages=[{"role": "user", "content": plot_prompt}],
        temperature=0.7,
        max_tokens=2000
    )
    
    print("Agent Response:")
    print("=" * 60)
    print(plot_response.choices[0].message.content)

## 9. Clean Up (Optional)

In [ ]:
async def delete_file(file_id: str):
    """Delete a file."""
    async with httpx.AsyncClient(timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        response = await http_client.delete(
            f"{AGENT_GATEWAY_URL}/v1/files/{file_id}",
            headers=headers
        )
        
        if response.status_code in [200, 204]:
            return {"success": True}
        else:
            return {"error": f"Delete failed: {response.status_code}"}

# Optional: Delete uploaded files
# Uncomment to clean up
# if "error" not in csv_upload:
#     result = await delete_file(csv_upload['id'])
#     if result.get('success'):
#         print(f"✅ Deleted CSV file: {csv_upload['id']}")
# 
# if "error" not in png_upload:
#     result = await delete_file(png_upload['id'])
#     if result.get('success'):
#         print(f"✅ Deleted PNG file: {png_upload['id']}")

print("Files preserved for future use.")
print("Uncomment deletion code to clean up.")

## Summary

You've learned how to:
- ✅ Upload various file types including images (PNG, JPEG)
- ✅ Process images through sandbox MCP server
- ✅ Validate PNG file structure and content
- ✅ Test security controls (authentication, path traversal)
- ✅ Reference files in agent conversations
- ✅ Generate visualizations from data
- ✅ Complete file lifecycle: upload → process → download

### Key Insights from Test Suite
- The agent_gateway serves as the file server
- Files are processed in sandbox with session isolation
- Proper file registration enables download access
- Security validations prevent unauthorized access
- PNG files can be processed with PIL/Pillow in sandbox

### Next Steps
- Try processing multiple images in batch
- Combine image and data analysis
- Explore advanced image transformations
- Test with different image formats (JPEG, GIF)
- Explore the `03_scientific_workflows.ipynb` notebook